# Mirage: external models, all predictions (Kaggle, T4 GPU)

Predictions only, no training (D39). Two ready-made Hugging Face models,
`roberta_large` (siebert/sentiment-roberta-large-english) and `distilbert_hf`
(distilbert-base-uncased-finetuned-sst-2-english), are run on **every evaluation set**: the SST-2
held-out / val / test / negation sets, the behavioral suite and the same 10,000 IMDb reviews
(plus their first + last sentence, `imdb_edges`) as every other model.

1. Turn on **GPU T4** (one GPU is enough) and **Internet** in the notebook settings.
2. **Save Version → Save & Run All**. About 20–30 minutes in total (download ~1.9 GB, then predictions; RoBERTa-large on the 10,000 reviews is most of it).
3. Download `external_outputs.zip` from the output panel and unzip it into the **repo root** on your
   laptop. It contains only `results/predictions/{roberta_large,distilbert_hf}__*.parquet` (14 files).

In [ ]:
REPO_URL = "https://github.com/AditeyaAItronics/mirage.git"

In [ ]:
import os, subprocess
os.chdir("/kaggle/working") if os.path.isdir("/kaggle/working") else None
if os.path.basename(os.getcwd()) == "mirage":
    os.chdir("..")
if os.path.isdir("mirage"):  # throwaway clone: always move it to the latest commit
    subprocess.run(["git", "-C", "mirage", "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", "mirage", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, "mirage"], check=True)
os.chdir("mirage")
print("code version:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -e ".[dl]"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU!")

In [ ]:
# Rebuilds exactly the same data as the laptop (fixed seeds), plus the behavioral suite.
!python -m mirage prepare
import pandas as pd
imdb = pd.read_parquet("artifacts/data/imdb.parquet")
print(len(imdb), "reviews; first ids:", imdb["item_id"].head(3).tolist())
!python -m mirage build-suite

In [ ]:
!python -m mirage predict-external --datasets sst2_heldout sst2_val sst2_test sst2_neg behavioral imdb imdb_edges

In [ ]:
import glob, zipfile
out = "/kaggle/working/external_outputs.zip" if os.path.isdir("/kaggle/working") else "external_outputs.zip"
files = [f for m in ("roberta_large", "distilbert_hf") for f in glob.glob(f"results/predictions/{m}__*.parquet")]
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)
print(out, os.path.getsize(out) // 2**20, "MB")
for f in files:
    p = pd.read_parquet(f)
    print(f, len(p), "rows, accuracy", round(p["correct"].mean(), 4))